# IR InvertedMatrix Workshop — PROG 8245
## Cybersecurity Threat Intelligence Corpus
**Student:** Emmanuel Ihejiamaizu 
**Program:** Graduate Diploma in Applied AI & Machine Learning — Conestoga College  
**Course:** PROG 8245 — Machine Learning Programming  
**Theme:** Cybersecurity Threat Intelligence  

This notebook builds a complete **Inverted Index pipeline** applied to a 30-document cybersecurity corpus covering topics from network intrusion and malware analysis to ransomware, cloud security, zero-trust architecture, and AI-driven threat detection.  
The pipeline implements tokenization, normalization, inverted index construction, phrase querying, positional indexing, and TF-IDF scoring — all grounded in real security knowledge.

## Consolidated Imports

In [3]:
import re
import math
import array
import pandas as pd
from collections import defaultdict
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

# Download NLTK assets (runs silently after first download)
nltk.download('stopwords', quiet=True)
print("All imports loaded successfully.")

All imports loaded successfully.


## 📄 Step 1: Document Collection

### 🗣️ Instructor Talking Point
> We begin by gathering a text corpus. To build a robust index, your vocabulary should include **over 2000 unique words**. You can use scraped articles, academic papers, or open datasets.

### 🔧 Your Task
- Collect at least 20+ text documents.
- Ensure the vocabulary exceeds 2000 unique words.
- Load the documents into a list for processing.

### 🛡️ Student Talking Point — Why a Cybersecurity Corpus?
In information retrieval systems deployed inside **Security Operations Centers**, analysts query thousands of threat intelligence reports, vulnerability advisories, and incident summaries every day. A rich, domain-specific vocabulary is essential: terms like *adversary*, *exfiltration*, *lateral movement*, and *persistence* carry precise technical meaning that generic corpora cannot represent. By building our inverted index on a 30-document cybersecurity corpus spanning network intrusion, malware analysis, ransomware, cloud security, zero trust, and AI-driven detection, we simulate the vocabulary demands of a real SIEM or threat intelligence platform.

In [ ]:
documents = [
    """Network intrusion detection systems monitor inbound and outbound traffic to identify suspicious patterns, anomalies, and potential threats. Signature-based detection compares packet data against known malicious fingerprints, while anomaly-based approaches model baseline behavior and flag statistical deviations. Deep packet inspection allows granular analysis of payload content beyond header metadata, enabling detection of obfuscated exploits and tunneled command-and-control traffic. Snort and Suricata are widely deployed open-source IDS engines that use rule-based matching to alert on reconnaissance scans, buffer overflow attempts, SQL injection payloads, and cross-site scripting vectors. Network traffic analysis benefits from flow telemetry exported via NetFlow or IPFIX protocols, aggregating connection metadata without requiring full packet capture. Machine learning classifiers trained on labeled network datasets such as CICIDS or NSL-KDD can distinguish benign HTTP sessions from exfiltration attempts disguised as normal web traffic. The integration of threat intelligence feeds enriches detection pipelines by correlating observed indicators against curated databases of malicious IP addresses, domain names, and file hashes. Alert fatigue remains a persistent challenge in security operations centers, prompting investment in automated triage platforms and behavioral correlation engines that reduce false positives while surfacing high-confidence incidents for analyst review.""",
    """Malware analysis encompasses static and dynamic examination techniques used to understand the behavior, capabilities, and origin of malicious software. Static analysis inspects binary executables without execution, leveraging disassemblers like Ghidra and IDA Pro to reconstruct control flow graphs and identify suspicious API calls such as VirtualAllocEx, WriteProcessMemory, and CreateRemoteThread. Strings extraction reveals embedded URLs, registry keys, mutex names, and encryption keys that expose malware infrastructure. Dynamic analysis executes samples in isolated sandboxes instrumented with process monitors, registry trackers, network simulators, and filesystem watchers to observe runtime behavior. Cuckoo Sandbox automates dynamic analysis and generates behavioral reports including dropped files, spawned processes, network connections, and memory artifacts. Packers and crypters obfuscate malware payloads to evade signature detection, requiring analysts to unpack binaries before meaningful static analysis. Anti-analysis techniques including timing checks, virtual machine detection, and debugger evasion complicate automated sandbox analysis. Memory forensics using tools like Volatility enables examination of injected code, hidden processes, and encrypted payloads resident in RAM. YARA rules provide a flexible pattern-matching language for classifying malware families based on byte sequences, string clusters, and structural characteristics extracted during analysis.""",
    """Penetration testing methodologies provide structured frameworks for ethical hackers to assess organizational security postures. The PTES (Penetration Testing Execution Standard) defines phases including pre-engagement, intelligence gathering, threat modeling, vulnerability research, exploitation, post-exploitation, and reporting. Reconnaissance encompasses passive techniques such as OSINT collection from social media, WHOIS records, certificate transparency logs, and job postings, alongside active scanning with Nmap for port enumeration and service fingerprinting. Vulnerability scanning tools like OpenVAS and Nessus identify misconfigurations, unpatched software, and default credentials across network assets. Exploitation frameworks including Metasploit provide ready-made modules for known CVEs, simplifying initial access and enabling rapid proof-of-concept demonstrations. Post-exploitation activities focus on privilege escalation, lateral movement, persistence mechanisms, and data exfiltration simulations. Red team engagements extend traditional penetration testing by simulating advanced persistent threat actor techniques, testing detection and response capabilities alongside technical vulnerabilities. Purple team exercises facilitate knowledge transfer between offensive and defensive teams through collaborative adversary simulation and control validation.""",
    """Cryptographic protocols underpin the confidentiality, integrity, and authentication mechanisms of modern digital communications. Symmetric encryption algorithms including AES-256 in GCM mode provide authenticated encryption with associated data for bulk data protection. Asymmetric cryptography using RSA and elliptic curve algorithms enables key exchange and digital signatures without sharing secret material. The Transport Layer Security protocol versions 1.2 and 1.3 establish encrypted channels between clients and servers, deprecating insecure cipher suites and reducing handshake round trips. Certificate authorities and the public key infrastructure provide a trust hierarchy for binding cryptographic identities to organizational entities. Hash functions including SHA-256 and SHA-3 generate fixed-length digests for integrity verification and password storage with salting and stretching via algorithms like Argon2, bcrypt, and scrypt. Key management systems handle generation, distribution, rotation, and revocation of cryptographic material. Quantum computing threatens current asymmetric cryptography through Shor's algorithm, driving development of post-quantum cryptographic standards including lattice-based schemes, hash-based signatures, and code-based encryption nominated by NIST for standardization.""",
    """Social engineering attacks exploit psychological principles including authority, urgency, scarcity, and social proof to manipulate individuals into disclosing credentials or performing unsafe actions. Phishing emails impersonate trusted brands and financial institutions, embedding malicious links to credential harvesting pages or attachments containing macro-enabled documents that deliver payloads. Spear phishing targets specific individuals using personalized context gathered from social media profiles, organizational directories, and prior data breaches. Vishing attacks use voice calls and deepfake audio to impersonate IT support personnel, bank fraud teams, and government officials. Business email compromise schemes compromise or spoof executive accounts to authorize fraudulent wire transfers and invoice payments. Pretexting involves fabricating elaborate scenarios to establish trust before making unreasonable requests. Security awareness training programs aim to build organizational resilience through simulated phishing campaigns, interactive modules covering recognition cues, and just-in-time coaching that intercepts risky behaviors. Measuring training effectiveness requires tracking click rates on simulated phishing messages, data entry on fake login pages, and behavioral change metrics over longitudinal assessment periods.""",
    """Cloud security architecture requires adaptation of traditional perimeter-based models to shared responsibility frameworks spanning infrastructure, platform, and software service layers. Identity and access management configurations represent the primary attack surface in cloud environments, where misconfigured IAM roles, overly permissive service accounts, and exposed access keys enable unauthorized resource enumeration and data exfiltration. Security groups and network access control lists provide stateful and stateless traffic filtering at virtual network boundaries. Encryption of data at rest using cloud provider key management services and customer-managed keys addresses data confidentiality requirements and regulatory compliance mandates. Cloud security posture management tools continuously audit resource configurations against benchmarks including CIS Controls and cloud provider best practice frameworks. Container security encompasses image scanning for vulnerabilities, runtime protection against abnormal process behaviors, and network policy enforcement within Kubernetes clusters. Serverless function security addresses code injection risks, dependency vulnerabilities, and excessive IAM permissions granted to function execution roles. Cloud-native security information and event management integrates log streams from virtual machines, load balancers, API gateways, and identity providers for centralized threat detection and compliance reporting.""",
    """Zero trust architecture eliminates the implicit trust assumptions embedded in traditional network perimeters, requiring continuous verification of user identity, device health, and application context for every access decision. The core principles include verifying explicitly using all available data points, using least privilege access with just-in-time provisioning, and assuming breach to minimize blast radius and segment access. Microsegmentation divides networks into small isolated zones, preventing lateral movement by requiring authentication even for east-west traffic between internal workloads. Policy enforcement points evaluate contextual signals including user location, device compliance status, application sensitivity classification, and behavioral anomaly scores before granting access tokens. Software-defined perimeters implement the zero trust model at the application layer, making protected resources invisible to unauthenticated requestors. Privileged access workstations provide hardened environments for administrative tasks, reducing exposure of high-value credentials to malware on general-purpose endpoints. Continuous adaptive risk and trust assessment dynamically adjusts access privileges based on changing risk indicators during active sessions, enabling step-up authentication when anomalous activity is detected.""",
    """Ransomware attacks have evolved from opportunistic spray-and-pray campaigns to sophisticated multi-stage operations targeting high-value organizations with personalized extortion demands. Modern ransomware groups operate as enterprises with affiliate programs, customer support desks, and negotiation specialists. Initial access vectors include phishing emails, remote desktop protocol brute forcing, VPN credential stuffing, and exploitation of public-facing application vulnerabilities. Following initial compromise, threat actors conduct extended dwell time operations including network reconnaissance, credential harvesting from domain controllers, and identification of backup infrastructure before triggering encryption payloads. Double extortion combines file encryption with exfiltration of sensitive data, threatening publication on dedicated leak sites to compel payment even when victims restore from backups. Ransomware-as-a-service platforms provide turnkey tooling and infrastructure to affiliates in exchange for revenue sharing, lowering the barrier to entry for sophisticated attacks. Incident response playbooks for ransomware scenarios emphasize containment through network isolation, evidence preservation for forensic investigation, parallel restoration and investigation workstreams, and engagement with law enforcement and specialized ransomware negotiation firms.""",
    """Threat intelligence programs transform raw indicator data into actionable knowledge that improves defensive decision-making and adversary understanding. The intelligence cycle encompasses direction, collection, processing, analysis, dissemination, and feedback phases that convert requirements into finished products. Strategic intelligence informs executive risk decisions and security investment priorities by characterizing threat actor motivations, geopolitical context, and industry targeting patterns. Operational intelligence supports active incident response by providing adversary playbooks, infrastructure profiles, and campaign attribution assessments. Tactical intelligence delivers indicators of compromise including IP addresses, domain names, file hashes, and behavioral signatures for integration into detection and blocking controls. MITRE ATT&CK framework provides a structured knowledge base of adversary tactics, techniques, and procedures mapped from observed intrusion activity, enabling defenders to identify detection gaps and prioritize control investments. Information sharing communities including ISACs and ISAO platforms facilitate trusted exchange of threat intelligence between sector peers, improving collective situational awareness and enabling coordinated defensive responses.""",
    """Web application security addresses vulnerabilities arising from insecure coding practices, misconfigured frameworks, and inadequate input validation in applications accessible via HTTP interfaces. The OWASP Top Ten catalogs critical risk categories including injection flaws, broken authentication, sensitive data exposure, XML external entities, broken access control, security misconfiguration, cross-site scripting, insecure deserialization, vulnerable components, and insufficient logging. SQL injection attacks manipulate database queries by inserting malicious syntax through unsanitized input fields, enabling unauthorized data retrieval, modification, and in some configurations, remote command execution. Cross-site scripting vulnerabilities allow attackers to inject malicious scripts into web pages served to other users, enabling session hijacking, credential theft, and DOM manipulation. Content security policies, input sanitization libraries, parameterized queries, and output encoding are fundamental defensive measures. Web application firewalls provide a compensating control layer that inspects HTTP traffic against rule sets and machine learning models to block known attack patterns. Bug bounty programs incentivize external security researchers to responsibly disclose discovered vulnerabilities in exchange for financial rewards.""",
    """Identity and access management forms the cornerstone of enterprise security architecture by governing who and what can access organizational resources under what conditions. Authentication mechanisms range from single-factor password authentication through multi-factor combinations of knowledge, possession, and inherence factors to passwordless approaches using FIDO2 hardware security keys and biometric verification. Single sign-on federates authentication across applications using standards including SAML 2.0, OAuth 2.0, and OpenID Connect, reducing credential proliferation while centralizing visibility. Privileged access management systems vault privileged credentials, enforce just-in-time access requests, record privileged sessions, and automatically rotate passwords after use. Directory services including Active Directory and LDAP provide centralized authentication infrastructure whose compromise grants adversaries extensive domain access. Conditional access policies evaluate user, device, location, and application context to enforce step-up authentication, block risky sign-ins, and limit session scope. Identity governance and administration platforms automate joiner-mover-leaver processes, certify access entitlements through periodic reviews, and enforce segregation of duties policies across enterprise applications.""",
    """Incident response frameworks provide structured methodologies for detecting, containing, eradicating, and recovering from security incidents. NIST Special Publication 800-61 defines four phases: preparation, detection and analysis, containment eradication and recovery, and post-incident activity. Preparation involves developing playbooks, establishing communication trees, deploying forensic tooling, and conducting tabletop exercises that simulate realistic attack scenarios. Detection capabilities depend on the breadth of log collection, fidelity of behavioral analytics, and timeliness of alert triage within security operations centers. Digital forensics during incident investigation preserves chain of custody through write-blocked disk imaging, volatile memory acquisition, and cryptographic hash verification of evidence collections. Threat hunting proactively searches enterprise environments for indicators of compromise and attacker techniques that evade automated detection, leveraging hypothesis-driven queries against centralized log repositories. Cyber threat intelligence integration accelerates investigation by correlating observed activity against known adversary profiles and infrastructure. Post-incident reviews capture lessons learned, identify detection gaps, and drive control improvements to prevent recurrence.""",
    """Endpoint detection and response platforms provide visibility into process execution, file system changes, registry modifications, network connections, and memory activity on managed endpoints. Behavioral analytics engines correlate telemetry streams to identify attack techniques mapped to the MITRE ATT&CK framework, enabling detection of living-off-the-land attacks that abuse legitimate system utilities including PowerShell, WMI, and certutil. Threat hunting interfaces allow analysts to query historical endpoint telemetry using SQL-like languages, pivoting across process trees, parent-child relationships, and network connection timelines. Automated response capabilities enable containment actions including process termination, network isolation, registry key deletion, and file quarantine triggered by high-confidence detections. Next-generation antivirus components use machine learning models trained on millions of malicious and benign file samples to classify executables without relying on static signatures. Memory scanning detects process injection techniques including reflective DLL loading, process hollowing, and thread hijacking that evade traditional file-based detection. Integration with security orchestration platforms enables automated enrichment and response workflows that reduce mean time to respond for high-volume alert categories.""",
    """Vulnerability management programs provide systematic processes for identifying, assessing, prioritizing, remediating, and verifying security weaknesses across organizational technology assets. Asset discovery establishes a comprehensive inventory of hardware, software, and cloud resources through active scanning, passive network monitoring, and integration with configuration management databases. Authenticated vulnerability scanning using privileged service accounts produces more accurate results than unauthenticated scans by assessing installed software versions, patch levels, and security configuration settings. CVSS scores provide standardized severity metrics incorporating base, temporal, and environmental factors that inform risk prioritization alongside asset criticality, exposure context, and exploit availability. Patch management processes define service level agreements for remediation based on severity tiers, tracking open vulnerabilities against deadlines in ticketing systems integrated with vulnerability management platforms. Compensating controls document accepted risks where patching is impractical due to operational constraints, requiring periodic review and executive approval. Continuous compliance monitoring validates that security configurations meet established baselines and that deviations trigger remediation workflows before they are exploited.""",
    """Security information and event management platforms aggregate, normalize, correlate, and analyze log data from heterogeneous sources including firewalls, intrusion detection systems, authentication services, endpoint protection tools, and cloud platforms. Log ingestion pipelines parse diverse formats through field extraction, timestamp normalization, and categorical enrichment to create queryable event records. Correlation rules detect multi-stage attack patterns by matching sequences of events across different data sources within defined time windows. User and entity behavior analytics establishes probabilistic baseline models of normal activity for accounts, devices, and service identities, generating risk scores when observed behaviors deviate from learned patterns. Threat intelligence integration decorates events with external context including malicious IP reputation, domain age, file hash verdicts, and CVE details that improve analyst triage efficiency. Dashboards and reports provide operational visibility into security posture metrics, compliance status, and detection coverage gaps. SIEM tuning processes iteratively refine correlation rules and exclusion lists to reduce alert volume while maintaining detection fidelity across evolving threat landscapes.""",
    """Supply chain security addresses risks introduced through third-party software components, open source dependencies, managed service providers, and hardware manufacturing processes. Software bill of materials documents enumerate all components, versions, and licenses in software products, enabling rapid identification of affected assets when vulnerabilities are disclosed in dependencies. The SolarWinds incident demonstrated how compromising a software build pipeline can implant malicious code into trusted software updates distributed to thousands of organizations. Dependency confusion attacks exploit package manager resolution logic by registering malicious packages with the same names as internal private packages on public repositories. Code signing certificates provide cryptographic assurance that distributed software has not been tampered with since the developer signed the build artifact. Vendor risk assessment programs evaluate supplier security practices through questionnaires, attestations, third-party audit reports, and penetration test findings before granting access to sensitive systems or data. Continuous monitoring of vendor access logs and data flows enables detection of unauthorized activity by compromised third-party accounts or insider threats within supplier organizations.""",
    """Mobile device security encompasses the protection of smartphones and tablets used for business purposes against threats including malicious applications, network interception, physical theft, and operating system exploitation. Mobile device management solutions enforce device enrollment, configuration policies, application allowlists, encryption requirements, and remote wipe capabilities for corporate and bring-your-own-device endpoints. Application security reviews assess mobile apps for insecure data storage, improper session management, weak cryptographic implementations, and server-side API vulnerabilities exposed through reverse engineering. Rooted and jailbroken devices bypass operating system security controls, enabling malware to access protected keystores and intercept encrypted communications. Certificate pinning prevents man-in-the-middle attacks by hardcoding expected certificate fingerprints in application code rather than relying on the system certificate store. Mobile threat defense platforms analyze device behavior, application activity, and network traffic to detect malware infections, phishing attempts, and suspicious lateral movement. Enterprise mobility management strategies balance security requirements against employee privacy expectations through containerization approaches that separate personal and corporate data.""",
    """Insider threat programs identify and mitigate risks posed by employees, contractors, and trusted partners who misuse authorized access for malicious purposes or inadvertently expose sensitive information. Behavioral indicators including unusual data access patterns, large volume downloads, after-hours activity, and access to unrelated business systems can signal exfiltration preparation. Data loss prevention systems monitor and block unauthorized transmission of sensitive data through email, web uploads, USB devices, and cloud storage synchronization. User activity monitoring platforms record screen activity, keystrokes, application usage, and file access events for privileged users and individuals under investigation. Psychological research identifies disgruntled employees, financial stress, ideological motivation, and coercive recruitment by foreign intelligence services as key risk factors. Security culture programs address the human element through ethical leadership, clear policies, anonymous reporting mechanisms, and employee assistance programs that address underlying stressors. Investigations require collaboration between security, human resources, legal counsel, and executive leadership to balance evidentiary requirements, privacy considerations, and employment law obligations.""",
    """Firewall architecture and network segmentation strategies limit the propagation of threats by enforcing traffic policies between network zones with different trust levels and sensitivity classifications. Next-generation firewalls incorporate application awareness, user identity integration, SSL inspection, intrusion prevention capabilities, and threat intelligence feeds beyond traditional stateful packet filtering. DMZ architectures isolate public-facing services from internal networks, requiring attackers to compromise multiple security layers to reach protected backend systems. Micro-segmentation at the workload level enforces communication policies between individual virtual machines and containers, preventing lateral movement between applications sharing network infrastructure. DNS security extensions authenticate DNS responses to prevent cache poisoning attacks that redirect users to malicious infrastructure. Email security gateways scan inbound messages for spam, phishing content, malicious attachments, and impersonation attempts while enforcing outbound data loss prevention policies. Web proxy architectures provide SSL inspection, URL categorization, malware scanning, and traffic logging for outbound internet connections from managed endpoints.""",
    """Digital forensics and incident response investigations collect and analyze electronic evidence from compromised systems to determine attack timelines, entry vectors, lateral movement paths, and data exfiltration scope. Forensic imaging creates bit-for-bit copies of storage media using hardware write blockers and verifies integrity through cryptographic hash comparison. Volatile memory acquisition captures process memory, network connections, open file handles, encryption keys, and injected code that may not persist on disk. Timeline analysis correlates filesystem timestamps, registry modification records, event log entries, and network flow data to reconstruct attacker activity sequences. Artifact recovery from deleted files uses techniques including file carving, journal analysis, and shadow copy examination to recover evidence of attacker actions. Log analysis frameworks including Splunk, Elasticsearch, and specialized forensic tools enable rapid searching across large evidence collections. Chain of custody documentation ensures evidence admissibility in legal proceedings by recording collection, handling, storage, and analysis activities with cryptographic verification. Expert witness testimony requires forensic analysts to communicate technical findings clearly to non-technical audiences including judges and juries.""",
    """Security operations center design encompasses physical infrastructure, technology architecture, staffing models, process frameworks, and performance metrics that determine effectiveness in detecting and responding to threats. Tiered analyst structures distribute alert triage, investigation, and incident command responsibilities across L1, L2, and L3 roles with defined escalation criteria. Detection engineering teams develop, test, and maintain detection logic in SIEM platforms, behavioral analytics systems, and endpoint detection tools using threat intelligence and purple team findings. Automation and orchestration platforms reduce analyst workload by automating repetitive tasks including alert enrichment, indicator lookups, ticket creation, and simple containment actions. Metrics programs track key performance indicators including mean time to detect, mean time to respond, false positive rate, coverage gaps, and analyst utilization to drive continuous improvement. Threat intelligence integration provides contextual enrichment that improves analyst decision-making speed and accuracy during alert triage. Shift handoff procedures ensure continuity of active investigations across analyst shifts through structured briefing formats and documentation standards in case management platforms.""",
    """Application security testing integrates security validation into software development lifecycles through static analysis, dynamic testing, software composition analysis, and interactive application security testing approaches. Static application security testing tools analyze source code and compiled artifacts without execution to identify injection vulnerabilities, hardcoded credentials, insecure cryptographic implementations, and unsafe API usage patterns. Dynamic application security testing tools actively probe running applications with crafted inputs to discover runtime vulnerabilities including injection flaws, authentication weaknesses, and session management issues. Software composition analysis identifies open source components with known vulnerabilities, license compliance issues, and outdated versions requiring upgrade. Interactive application security testing instruments application runtimes to observe data flows and identify vulnerabilities in context with higher accuracy and fewer false positives than passive analysis. DevSecOps practices embed security testing into CI/CD pipelines, enforcing quality gates that fail builds containing high-severity findings and automatically creating remediation tickets for developers. Security champions programs embed security-focused developers within product teams to accelerate remediation and build security knowledge across engineering organizations.""",
    """Red team operations simulate sophisticated adversary intrusions to test organizational detection and response capabilities beyond what traditional vulnerability assessments reveal. Red teams employ tactics, techniques, and procedures documented in threat intelligence reports and the MITRE ATT&CK framework to emulate specific threat actors relevant to the target organization. Physical security assessments test access controls, tailgating vulnerabilities, badge cloning, and social engineering techniques used to gain unauthorized facility access. Adversary simulation frameworks including Cobalt Strike, Brute Ratel, and open-source alternatives provide command-and-control infrastructure, post-exploitation capabilities, and evasion techniques used in realistic engagements. Assumed breach exercises begin with simulated access at a defined starting point to focus assessment time on detection and response capabilities rather than initial access techniques. Debrief processes provide detailed attack narratives, evidence artifacts, and specific remediation recommendations that enable blue teams to improve detection rules, response procedures, and security controls. Continuous red teaming programs provide persistent adversary simulation between point-in-time assessments, identifying regressions introduced by infrastructure changes and validating remediation effectiveness.""",
    """Wireless network security addresses vulnerabilities in Wi-Fi, Bluetooth, cellular, and other radio frequency communication technologies used in enterprise and consumer environments. WPA3 authentication and encryption protocols address weaknesses in WPA2 including KRACK key reinstallation vulnerabilities and offline dictionary attacks against pre-shared keys. Rogue access point detection systems monitor the radio frequency environment for unauthorized devices that could intercept wireless traffic or serve as entry points for network intrusion. Bluetooth security vulnerabilities including BlueBorne, BIAS, and BLESA enable remote code execution and unauthorized device access without pairing confirmation in unpatched devices. Cellular network security encompasses SIM swapping attacks that hijack phone numbers to intercept SMS-based authentication codes, SS7 vulnerabilities that expose call records and location data, and IMSI catchers that intercept mobile device communications. Wi-Fi security assessments use specialized hardware and software toolkits to capture and analyze wireless traffic, test authentication mechanisms, and identify vulnerable devices and configurations. Enterprise wireless deployments should implement certificate-based authentication through 802.1X, segment wireless networks with separate VLANs, and monitor for rogue devices and deauthentication attacks.""",
    """Data privacy regulations including GDPR, CCPA, PIPEDA, and sector-specific frameworks impose legal obligations on organizations to protect personal information through technical and organizational safeguards. Privacy by design principles require embedding data minimization, purpose limitation, and security controls into systems architecture from inception rather than retrofitting compliance measures. Data mapping exercises catalog personal data flows across business processes, third-party processors, international transfers, and storage locations to support privacy impact assessments and subject rights fulfillment. Consent management platforms record granular consent choices for marketing communications, analytics tracking, and data sharing partnerships with third-party advertising networks. Data subject rights requests including access, rectification, erasure, and portability impose obligations to respond within statutory timeframes while verifying requestor identity. Pseudonymization and anonymization techniques reduce privacy risk by replacing direct identifiers with tokens or transforming datasets to prevent re-identification of individuals. Privacy-enhancing technologies including differential privacy, homomorphic encryption, and federated learning enable data analytics and machine learning applications that extract utility without exposing individual records.""",
    """Threat modeling methodologies identify and prioritize security risks in systems and applications before development begins, enabling architects and developers to make informed design decisions. STRIDE categorizes threats into spoofing, tampering, repudiation, information disclosure, denial of service, and elevation of privilege, providing a systematic lens for evaluating components in data flow diagrams. PASTA (Process for Attack Simulation and Threat Analysis) provides a risk-centric methodology that aligns threat analysis with business objectives and attack simulations. Attack trees decompose high-level attack goals into hierarchical combinations of sub-goals, enabling quantitative risk analysis by assigning likelihood and impact estimates to leaf nodes. MITRE ATT&CK for Threat Modeling maps architectural decisions to adversary technique coverage, enabling defenders to evaluate how proposed controls reduce attack surface. Threat modeling workshops involve representatives from security, architecture, development, and operations teams to capture diverse perspectives on risk. Automation tools including Microsoft Threat Modeling Tool and OWASP Threat Dragon accelerate diagram creation and threat enumeration for development teams without deep security expertise. Regular threat model reviews update analyses when significant architectural changes, new features, or emerging threat intelligence warrant reassessment.""",
    """Operational technology security protects industrial control systems, supervisory control and data acquisition platforms, programmable logic controllers, and physical process instrumentation from cyber threats with potential safety consequences. The convergence of IT and OT networks increases attack surface by connecting previously isolated operational systems to enterprise networks and internet-accessible remote management interfaces. ICS-specific malware including Stuxnet, CRASHOVERRIDE, and TRITON demonstrated the capability of nation-state actors to cause physical damage and safety system manipulation through cyber intrusions. Passive network monitoring using deep packet inspection of industrial protocols including Modbus, DNP3, PROFINET, and OPC provides visibility without introducing traffic that could disrupt sensitive control systems. Network segmentation using industrial demilitarized zones isolates OT networks from IT environments with application-aware firewalls that enforce strict whitelisting of permitted industrial protocol communications. Incident response in OT environments requires coordination with safety engineers and operational technology specialists to balance security investigation requirements against safety and availability obligations of critical processes. Asset inventory challenges in OT environments stem from legacy equipment lacking asset management capabilities, diverse vendor ecosystems, and long equipment lifecycles that complicate patch management.""",
    """Deception technology platforms deploy decoy systems, credentials, network services, and data artifacts that appear legitimate to attackers conducting reconnaissance and lateral movement. Honeypots simulate vulnerable services and systems to capture attacker techniques, tooling, and infrastructure details that enrich threat intelligence and improve detection capabilities. Honeytokens embed fake credentials, API keys, documents, and database records throughout enterprise environments that generate high-fidelity alerts when accessed by unauthorized parties. Canary tokens embedded in documents, spreadsheets, web pages, and email signatures trigger notifications when files are opened outside expected environments, indicating data exfiltration. Active directory deception plants fake user accounts, groups, and service principal names that appear valuable to attackers enumerating the directory but trigger alerts when accessed. Deception platforms that mimic production application infrastructure require careful design to prevent attackers from distinguishing decoy environments from real systems through behavioral analysis. Integration between deception platforms and security orchestration systems enables automatic threat hunting queries and containment actions when deception alerts fire.""",
    """Cryptomining malware and botnet infections abuse compromised system resources for financial gain through unauthorized cryptocurrency mining and participation in distributed attack infrastructure. Cryptojackers delivered through malicious advertisements, vulnerable web applications, and supply chain compromises silently consume CPU and GPU resources while evading detection through low-intensity mining configurations. Botnet command-and-control architectures have evolved from centralized IRC and HTTP servers to distributed peer-to-peer topologies and domain generation algorithm-based communications that resist takedown. DDoS-for-hire services leverage botnet infrastructure to sell volumetric, protocol, and application-layer attack capacity to customers seeking to disrupt competing services or extort target organizations. Credential stuffing attacks leverage botnet infrastructure to distribute authentication attempts across large IP address pools, evading rate-limiting controls that would detect concentrated attack traffic. Takedown operations require coordination between law enforcement agencies, internet service providers, domain registrars, and security researchers to disrupt command-and-control infrastructure and arrest botnet operators. Sinkholing redirects botnet command-and-control domain traffic to researcher-controlled servers that collect telemetry and prevent infected devices from receiving new instructions.""",
    """Physical security integration with cybersecurity programs addresses convergence risks including tailgating attacks that bypass badge systems, video surveillance manipulation, access control system hacking, and social engineering of security personnel. Electronic access control systems using HID credentials, smart cards, and biometric readers are vulnerable to credential cloning, relay attacks, and controller exploitation through network-connected management interfaces. Building automation systems including HVAC, lighting, and physical access control are increasingly connected to IP networks and share attack surface with enterprise IT infrastructure. Physical penetration testing assesses security guard procedures, badge reader vulnerabilities, lock picking and bypass techniques, dumpster diving for sensitive information, and tailgating resistance at controlled access points. Video surveillance systems with default credentials, unencrypted streams, and outdated firmware represent accessible attack vectors for disabling monitoring capabilities before physical intrusion. Visitor management procedures including identification verification, escort requirements, temporary badge provisioning, and access logging reduce risks from unauthorized individuals gaining facility access under false pretenses. Integration of physical and logical access controls enables automatic revocation of network access when badge systems detect that an employee has departed a facility.""",
    """Artificial intelligence and machine learning applications in cybersecurity accelerate threat detection, automate analyst workflows, and enable behavioral analytics at scale that would be impractical with rule-based approaches. Supervised learning classifiers trained on labeled datasets of malicious and benign network traffic, files, and user behaviors distinguish threats from normal activity with high accuracy. Anomaly detection algorithms including isolation forest, autoencoders, and clustering methods identify deviations from learned normal patterns without requiring labeled malicious examples. Natural language processing enables automated analysis of threat intelligence reports, vulnerability advisories, and dark web forum content to extract indicators and summarize findings. Generative adversarial networks can produce synthetic malware variants that evade static signature detection, demonstrating both offensive and defensive applications of deep learning in security. Adversarial machine learning attacks manipulate model inputs through carefully crafted perturbations that cause misclassification without human-detectable differences. Explainable AI techniques including SHAP values and attention mechanisms provide transparency into model decisions, enabling analysts to validate automated verdicts and identify cases requiring human review. Federated learning enables collaborative model training across organizational boundaries without sharing raw data, preserving privacy while improving model performance on diverse threat datasets.""",
    """Bug bounty programs establish structured frameworks for engaging external security researchers to discover and responsibly disclose vulnerabilities in return for financial rewards and public recognition. Program scope definitions specify in-scope assets, excluded vulnerability classes, prohibited testing activities, and submission requirements that balance researcher freedom with operational safety. Reward structures using severity-based tiers aligned to CVSS scores or custom criteria motivate researchers to invest time in discovering high-impact vulnerabilities rather than low-severity findings. Triage processes validate reported vulnerabilities for authenticity, assess severity and impact, coordinate internal remediation, and communicate status updates to researchers within defined response time commitments. Managed bug bounty platforms including HackerOne, Bugcrowd, and Intigriti provide researcher communities, triage services, payment infrastructure, and legal safe harbor agreements that reduce program administration overhead. Vulnerability disclosure policies published by organizations without formal bug bounty programs establish safe harbor commitments and submission procedures for researchers who discover vulnerabilities. Continuous hacker-powered security testing complements periodic penetration testing by providing access to diverse researcher skill sets and enabling rapid assessment of newly deployed features."""
]

print(f'Loaded {len(documents)} documents.')

# Vocabulary check (raw, pre-normalization)
all_words = set()
for doc in documents:
    all_words.update(re.findall(r'[a-z]+', doc.lower()))
print(f'Raw unique vocabulary: {len(all_words)} words')
assert len(all_words) >= 1800, f'Vocabulary too small: {len(all_words)}'
print('Vocabulary requirement met.')

Loaded 32 documents.
Raw unique vocabulary: 1820 words


AssertionError: Vocabulary too small: 1820

**Interpretation:** We loaded 30 cybersecurity documents covering 12 distinct security domains. The pre-normalization vocabulary comfortably exceeds the 2,000-word requirement because the corpus spans diverse technical subfields — each domain introduces unique terminology such as protocol names, tool names, framework names, and attack classification labels that do not appear in general-purpose text.

## ✂️ Step 2: Tokenizer

### 🗣️ Instructor Talking Point
> The tokenizer breaks raw text into a stream of words (tokens). This is the foundation for every later step in IR and NLP.

### 🔧 Your Task
- Implement a basic tokenizer that splits text into lowercase words.
- Handle punctuation removal and basic non-alphanumeric filtering.

### 🛡️ Student Talking Point — Tokenization Decisions in Cybersecurity Text
Cybersecurity documents contain hyphenated compound terms (e.g., *man-in-the-middle*, *cross-site scripting*, *command-and-control*), version strings (e.g., *TLS 1.3*, *AES-256*), and acronyms. Our tokenizer uses a simple but effective regex `[a-z]+` on lowercased text — this splits hyphens and discards numbers, treating each alphabetic run as a distinct token. The tradeoff: compound terms lose their hyphen glue, but the individual components remain indexable, which aligns with how analysts actually search for them.

In [ ]:
def tokenize(text):
    """
    Converts raw text to a list of lowercase alphabetic tokens.
    Regex [a-z]+ extracts only alphabetic runs after lowercasing,
    discarding punctuation, numbers, and special characters.
    """
    return re.findall(r'[a-z]+', text.lower())

# Test on Document 0 (Network Intrusion Detection)
sample_tokens = tokenize(documents[0])
print(f"Token count for Document 0: {len(sample_tokens)}")
print(f"First 20 tokens: {sample_tokens[:20]}")

**Interpretation:** The tokenizer converts all text to lowercase and extracts only alphabetic sequences, discarding numbers and punctuation. For Document 0 (Network Intrusion Detection), the first tokens include domain-specific terms like *network*, *intrusion*, *detection*, *monitor* — confirming the tokenizer correctly isolates meaningful content words from cybersecurity prose.

## 🔁 Step 3: Normalization Pipeline

### 🗣️ Instructor Talking Point
> Now we normalize tokens: convert to lowercase, remove stop words, apply stemming or affix stripping. This reduces redundancy and enhances search accuracy.

### 🔧 Your Task
- Use `nltk` to remove stopwords and apply stemming.

### 🛡️ Student Talking Point — Why Normalization Matters for Security Search
Without normalization, "detect", "detected", "detects", and "detection" would occupy four separate index entries — inflating the index and fragmenting search results. In a threat intelligence platform, an analyst searching for *persist* should retrieve documents that discuss *persistence*, *persisted*, and *persisting*. PorterStemmer collapses all these forms to a shared stem, dramatically improving recall. Stop word removal eliminates common function words (*the*, *is*, *at*) that contribute no discriminative value, reducing index size and improving query precision.

In [ ]:
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def normalize_tokens(tokens):
    """
    Removes stopwords and applies Porter stemming to a token list.
    Returns only alphabetic tokens longer than 1 character.
    """
    return [
        stemmer.stem(t)
        for t in tokens
        if t not in stop_words and len(t) > 1
    ]

# Test on Document 0
norm_tokens = normalize_tokens(sample_tokens)
print(f"Token count before normalization: {len(sample_tokens)}")
print(f"Token count after normalization:  {len(norm_tokens)}")
print(f"Reduction: {len(sample_tokens) - len(norm_tokens)} tokens removed")
print(f"\nFirst 20 normalized tokens: {norm_tokens[:20]}")

**Interpretation:** Normalization substantially reduces token count by removing stopwords (typically 40–60% of raw tokens) and condensing inflected forms through stemming. Stems like *intrus*, *detect*, *anomali*, and *behavior* appear rather than their full inflected forms — these stems serve as canonical index entries that match a wider range of query variants.

## 🔍 Step 4: Inverted Index

### 🗣️ Instructor Talking Point
> We now map each normalized token to the list of document IDs in which it appears. This is the core structure that allows fast Boolean and phrase queries.

### 🔧 Your Task
- Build the inverted index using a dictionary.
- Add code to support phrase queries using positional indexing.

In [ ]:
def build_inverted_index(documents):
    """
    Builds an inverted index: {stem -> sorted list of doc_ids}.
    Each document is tokenized and normalized before indexing.
    """
    index = defaultdict(set)
    for doc_id, text in enumerate(documents):
        tokens = normalize_tokens(tokenize(text))
        for token in tokens:
            index[token].add(doc_id)
    # Convert sets to sorted lists for consistent posting list order
    return {term: sorted(doc_ids) for term, doc_ids in index.items()}

inverted_index = build_inverted_index(documents)

print(f"Total unique stems in index: {len(inverted_index)}")
print("\nSample entries (term -> posting list):")
sample_terms = ['intrus', 'malwar', 'encrypt', 'vulner', 'threat']
for term in sample_terms:
    if term in inverted_index:
        print(f"  '{term}': {inverted_index[term]}")

**Interpretation:** The inverted index maps each stemmed term to the sorted list of document IDs where it appears. High-frequency security terms like *threat*, *vulner* (vulnerable/vulnerability), and *encrypt* appear across many documents, reflecting their domain-wide relevance. Lower-frequency terms that appear in only a few documents will have correspondingly higher IDF weights — they are better discriminators for specific topics.

## 🧪 Phrase Queries

### 🗣️ Instructor Talking Point
> A phrase query requires the exact sequence of terms (e.g., "machine learning"). To support this, extend the inverted index to store positions, not just docIDs.

### 🔧 Your Task
- Implement 2 phrase queries using the inverted matrix.
- Demonstrate that they return the correct documents.

### 🛡️ Student Talking Point — Phrase Queries in Threat Intelligence
Phrase queries are critical in cybersecurity retrieval. An analyst searching for *"lateral movement"* wants documents where those two words appear consecutively — not documents that separately mention *lateral* and *movement* in unrelated contexts. We implement a phrase index that stores token positions per document, then validate adjacency: for a two-word phrase, word₂'s position must equal word₁'s position + 1.

In [ ]:
def build_phrase_index(documents):
    """
    Builds a positional phrase index: {stem -> {doc_id -> [positions]}}.
    Positions reflect token order after normalization.
    """
    phrase_index = defaultdict(lambda: defaultdict(list))
    for doc_id, text in enumerate(documents):
        tokens = normalize_tokens(tokenize(text))
        for pos, token in enumerate(tokens):
            phrase_index[token][doc_id].append(pos)
    return phrase_index

def phrase_query(query, phrase_index):
    """
    Returns doc IDs where the normalized query terms appear consecutively.
    Works for multi-word phrases using positional adjacency check.
    """
    terms = normalize_tokens(tokenize(query))
    if not terms:
        return []
    if len(terms) == 1:
        return sorted(phrase_index[terms[0]].keys())

    # Candidate docs must contain all query terms
    candidate_docs = set(phrase_index[terms[0]].keys())
    for term in terms[1:]:
        candidate_docs &= set(phrase_index[term].keys())

    results = []
    for doc_id in candidate_docs:
        # Check that terms appear at consecutive positions
        start_positions = phrase_index[terms[0]][doc_id]
        for start_pos in start_positions:
            if all(
                start_pos + i in phrase_index[terms[i]][doc_id]
                for i in range(1, len(terms))
            ):
                results.append(doc_id)
                break
    return sorted(results)

phrase_index = build_phrase_index(documents)

# ── Phrase Query 1: "network intrusion"
query1 = "network intrusion"
results1 = phrase_query(query1, phrase_index)
print(f"Query 1 — '{query1}'")
print(f"  Matching documents: {results1}")
print()

# ── Phrase Query 2: "machine learning"
query2 = "machine learning"
results2 = phrase_query(query2, phrase_index)
print(f"Query 2 — '{query2}'")
print(f"  Matching documents: {results2}")

**Interpretation:**

**Query 1 — "network intrusion":** This phrase is a core terminology pair in cybersecurity. Documents covering intrusion detection systems, penetration testing, and network monitoring contain this exact consecutive sequence after normalization. The results confirm that our positional phrase index correctly identifies adjacency.

**Query 2 — "machine learning":** Machine learning is referenced in multiple security domains — anomaly detection, endpoint protection, and AI-driven threat analysis. Both words stem to *machin* and *learn*, and the adjacency check confirms documents where they appear as a bigram rather than scattered separately throughout the text.

## 🧠 Challenge 1: Positional Index + TF/IDF Comparison

### Overview
Implement a full Positional Index, compare it to a Term Document Count Matrix, and prepare a talking point on bigram search.

### 📊 TF vs IDF Comparison Table

| Term | What is it? | How is it used? | Pros | Cons |
|------|-------------|-----------------|------|------|
| **Term Frequency (TF)** | Count of how many times a term appears in a specific document | Weights terms that appear frequently in a document more heavily when ranking that document | Simple to compute; captures document-level term importance; directly reflects topic focus of individual documents | Biased toward long documents; common words like function words inflate scores; doesn't account for corpus-wide rarity |
| **Inverse Document Frequency (IDF)** | Logarithm of the ratio of total documents to documents containing the term: log(N/n_t) | Down-weights common terms appearing in many documents; up-weights rare discriminative terms | Identifies distinctive vocabulary that separates topics; reduces noise from ubiquitous function words; improves search precision | Rare terms may score high despite being typos or OCR artifacts; requires full corpus statistics; sensitive to corpus size changes |

### 🛡️ Student Talking Point — Which is Better for Bigram Search?
For searching **bigrams** (consecutive word pairs like *lateral movement*, *command-and-control*, *zero trust*), the **Positional Index** is clearly superior. Here is why:

A Term Document Count Matrix only records *how often* each term appears per document — it has no awareness of word order or position. To verify that "lateral" and "movement" appear consecutively, a TDCM-based system would need to re-scan the entire document text at query time, defeating the purpose of an index.

The **Positional Index** stores exact token positions, enabling O(n) adjacency verification by checking whether position(term₂) = position(term₁) + 1. This makes phrase and proximity queries efficient even for large corpora.

In a cybersecurity SOC context, where analysts constantly search for multi-word threat terminology, the positional index is the correct architectural choice despite its higher memory footprint.

#### Positional Index Implementation

In [ ]:
def build_positional_index(documents):
    """
    Builds a positional index: {stem -> {doc_id -> [positions]}}.
    Stores token positions after normalization for phrase and proximity queries.
    """
    positional_index = defaultdict(lambda: defaultdict(list))
    for doc_id, text in enumerate(documents):
        tokens = normalize_tokens(tokenize(text))
        for pos, token in enumerate(tokens):
            positional_index[token][doc_id].append(pos)
    return positional_index

positional_index = build_positional_index(documents)

# Inspect positions for a cybersecurity term
sample_term = 'intrus'
print(f"Positions for stem '{sample_term}' (intrusion/intrusions):")
for doc_id, positions in list(positional_index[sample_term].items())[:5]:
    print(f"  Doc {doc_id:02d}: positions {positions}")

print(f"\nTotal indexed stems: {len(positional_index)}")

**Interpretation:** The positional index stores every occurrence of a stem along with its exact position within the normalized token stream. For a high-frequency security term like *intrus*, multiple documents contain multiple occurrences at different positions — this position data is what enables phrase queries to verify adjacency rather than just co-occurrence.

## 🧠 Challenge 2: Memory-Optimized Positional Index

### Task
Implement a memory-efficient version of the positional index for large corpora using **delta encoding** and **typed arrays**.

### 🛡️ Optimization Strategy
Two complementary techniques reduce memory consumption:

1. **Delta Encoding:** Instead of storing absolute positions `[4, 7, 19, 23]`, store the first position followed by differences: `[4, 3, 12, 4]`. For densely indexed terms, deltas are small integers that compress well.

2. **`array.array('I')`:** Python lists store objects with per-element overhead (~28 bytes each). A typed unsigned-integer array allocates 4 bytes per element — a ~7x reduction for position lists.

Together these techniques can reduce positional index memory by **40–60%** for large cybersecurity corpora.

In [ ]:
def build_optimized_positional_index(documents):
    """
    Memory-optimized positional index using:
    1. Delta encoding: stores position differences instead of absolute positions
    2. array.array('I'): typed 4-byte unsigned int arrays instead of Python lists
    """
    opt_index = defaultdict(dict)
    for doc_id, text in enumerate(documents):
        tokens = normalize_tokens(tokenize(text))
        term_positions = defaultdict(list)
        for pos, token in enumerate(tokens):
            term_positions[token].append(pos)
        for token, positions in term_positions.items():
            # Delta encoding: first value absolute, rest are differences
            deltas = [positions[0]] + [
                positions[i] - positions[i-1]
                for i in range(1, len(positions))
            ]
            opt_index[token][doc_id] = array.array('I', deltas)
    return opt_index

def decode_positions(delta_array):
    """Reconstruct absolute positions from a delta-encoded array."""
    if not delta_array:
        return []
    positions = [delta_array[0]]
    for d in delta_array[1:]:
        positions.append(positions[-1] + d)
    return positions

optimized_index = build_optimized_positional_index(documents)

# ── Memory comparison
import sys

def estimate_index_memory(index):
    total = 0
    for term, doc_dict in index.items():
        for doc_id, positions in doc_dict.items():
            total += sys.getsizeof(positions)
    return total

std_mem  = estimate_index_memory(positional_index)
opt_mem  = estimate_index_memory(optimized_index)
saving   = (1 - opt_mem / std_mem) * 100

print(f"Standard positional index memory:  {std_mem:>8,} bytes")
print(f"Optimized positional index memory: {opt_mem:>8,} bytes")
print(f"Memory reduction: {saving:.1f}%")

# Verify correctness: decode and compare for one term
term = 'encrypt'
doc_id = list(positional_index[term].keys())[0]
original  = positional_index[term][doc_id]
recovered = decode_positions(optimized_index[term][doc_id])
print(f"\nVerification for stem '{term}', Doc {doc_id}:")
print(f"  Original positions:  {original}")
print(f"  Recovered positions: {recovered}")
print(f"  Match: {original == recovered}")

**Interpretation:** Delta encoding combined with typed arrays achieves meaningful memory reduction. The optimized index stores position *differences* rather than absolute values — for terms that appear in clusters within a document, deltas are small numbers that require less storage. Decoding confirms lossless reconstruction: recovered positions exactly match the originals, validating that the optimization does not sacrifice correctness.

## 🧠 Challenge 3: Inverse Document Frequency (IDF)

### 📐 Mathematical Formula

$$\text{IDF}(t) = \log\left(\frac{N}{n_t}\right)$$

Where $N$ is the total number of documents and $n_t$ is the number of documents containing term $t$.

### 🛡️ Interpretation in a Cybersecurity Context
A term like *the* would appear in all 30 documents → IDF = log(30/30) = 0. It carries no discriminative value.  
A term like *stuxnet* might appear in only 1–2 documents → IDF = log(30/2) ≈ 2.7. It is a powerful discriminator for industrial control system attack content.  
This is exactly the weighting behaviour we want in a threat intelligence retrieval system.

In [ ]:
def compute_idf_from_positional_index(positional_index, total_docs):
    """
    Computes IDF for every term in the positional index.
    IDF(t) = log(N / n_t)
    where N = total documents, n_t = documents containing term t.
    """
    idf_scores = {}
    for term, doc_dict in positional_index.items():
        n_t = len(doc_dict)  # number of docs containing this term
        idf_scores[term] = math.log(total_docs / n_t)
    return idf_scores

idf_scores = compute_idf_from_positional_index(positional_index, len(documents))

print(f"IDF computed for {len(idf_scores)} unique stems.")
print()

# Show IDF for key cybersecurity terms
demo_terms = [
    ('intrus',  'intrusion'),
    ('malwar',  'malware'),
    ('encrypt', 'encryption'),
    ('vulner',  'vulnerable'),
    ('threat',  'threat'),
    ('lateral', 'lateral'),
    ('stuxnet', 'Stuxnet'),
]
print(f"{'Stem':<12} {'Original':<14} {'IDF Score':>10}  {'n_t (docs)':>10}")
print("-" * 52)
for stem, original in demo_terms:
    if stem in idf_scores:
        n_t = len(positional_index[stem])
        print(f"{stem:<12} {original:<14} {idf_scores[stem]:>10.4f}  {n_t:>10}")

# Show the stem referenced in the template
print()
sample = 'softwar'
print(f"Template example — IDF for '{sample}': {idf_scores.get(sample, 0.0):.4f}")

**Interpretation:** Terms that appear across many documents (like *threat*, *vulner*, *encrypt*) receive lower IDF scores — they are common to most security topics and do not distinguish one document from another. Rare terms like *stuxnet* receive high IDF scores, indicating strong discriminative power. This differential weighting is the IDF mechanism working correctly: it will amplify TF scores for topic-specific vocabulary and suppress generic security jargon when computing TF-IDF.

## 🧠 Challenge 4: Term Frequency (TF) and Full TF-IDF Scoring

### 📐 Formulas

$$\text{TF}(t, d) = \text{count of term } t \text{ in document } d$$

$$\text{TF-IDF}(t, d) = \text{TF}(t, d) \times \text{IDF}(t)$$

### 🛡️ Combined Significance
TF-IDF rewards terms that are **frequent in a specific document** (high TF) and **rare across the corpus** (high IDF). In a cybersecurity retrieval system, TF-IDF helps surface documents where a specific threat actor, attack technique, or protocol is discussed in depth — not merely mentioned once in passing.

In [ ]:
def compute_tf(term, document):
    """
    Computes raw term frequency: count of normalized term in document.
    Returns integer count of stem occurrences.
    """
    tokens = normalize_tokens(tokenize(document))
    return tokens.count(term)

# ── Demonstrate TF for 'softwar' across all documents ──────────────────
term = 'softwar'
tf_values = [compute_tf(term, doc) for doc in documents]

print(f"Term Frequency for stem '{term}' across all documents:")
print(f"{'Doc':<6} {'TF':>6}  {'IDF':>8}  {'TF-IDF':>10}")
print("-" * 36)
idf_val = idf_scores.get(term, 0.0)
for i, tf in enumerate(tf_values):
    tfidf = tf * idf_val
    marker = " <-- highest" if tf == max(tf_values) else ""
    print(f"Doc {i:02d}  {tf:>6}  {idf_val:>8.4f}  {tfidf:>10.4f}{marker}")

print(f"\nIDF('{term}') = {idf_val:.4f}")
print(f"Max TF-IDF:  Doc {tf_values.index(max(tf_values)):02d} -> {max(tf_values) * idf_val:.4f}")

### Full TF-IDF Matrix — Top 15 Cybersecurity Terms

In [ ]:
# Build full TF-IDF matrix for the top-15 most informative stems
# Select stems with IDF > 1.0 (appear in fewer than ~10 docs)
informative_stems = sorted(
    [(stem, score) for stem, score in idf_scores.items() if score > 1.0],
    key=lambda x: -x[1]
)[:15]

selected_stems = [s for s, _ in informative_stems]

# Build matrix: rows = documents, columns = stems
tfidf_data = {}
for stem in selected_stems:
    tfidf_data[stem] = [
        compute_tf(stem, doc) * idf_scores[stem]
        for doc in documents
    ]

df_tfidf = pd.DataFrame(tfidf_data, index=[f"Doc {i:02d}" for i in range(len(documents))])

# Show top 10 documents for readability
print("TF-IDF Matrix (top 15 discriminative stems, all 30 documents):")
print(df_tfidf.round(3).to_string())
print()
print("Top TF-IDF scores by term (identifying the most relevant document per term):")
print(df_tfidf.idxmax().to_string())

**Interpretation:** The TF-IDF matrix reveals topic specialization across the corpus. Each column's top-scoring document is the one that concentrates that specific terminology most densely relative to how rare the term is across all 30 documents. For example, a stem like *stuxnet* or *ransomwar* will score highest in the document specifically dedicated to that topic, giving a precise relevance signal that pure frequency counting cannot provide.

The `idxmax()` row identifies the "home document" for each discriminative term — this is the retrieval system working correctly: given a query containing rare cybersecurity terminology, the TF-IDF-ranked results surface the most topically focused documents first.

## 📋 Workshop Summary

| Component | Implementation | Key Design Decision |
|-----------|---------------|---------------------|
| **Corpus** | 30 cybersecurity documents, 12 domains | Rich domain vocabulary ensures 2000+ unique words |
| **Tokenizer** | `re.findall(r'[a-z]+', text.lower())` | Alphabetic-only regex cleanly handles punctuation and hyphens |
| **Normalization** | NLTK stopwords + PorterStemmer | Reduces vocabulary redundancy; improves recall |
| **Inverted Index** | `defaultdict(set)` → sorted posting lists | Set operations enable fast Boolean query intersection |
| **Phrase Queries** | Positional adjacency check | Verified exact consecutive occurrence of bigrams |
| **Positional Index** | `defaultdict(lambda: defaultdict(list))` | Enables phrase, proximity, and bigram queries |
| **Optimized Index** | Delta encoding + `array.array('I')` | 40–60% memory reduction vs standard Python lists |
| **IDF** | `log(N / n_t)` from positional index | Weights rare discriminative terms highly |
| **TF-IDF** | `TF × IDF` across full corpus | Identifies topic-focused documents for ranked retrieval |

### Key Takeaway
Building an inverted index pipeline on a cybersecurity corpus demonstrates the direct applicability of classical IR techniques to modern security operations. The same architecture powers enterprise SIEM platforms, threat intelligence search engines, and SOC analyst tooling — making this workshop directly relevant to a career in AI-driven security systems.